# UCS420: Cognitive Computing — Assignment 7 (NLP-I)

**Name:** Ali Ekram  |  **Roll No:** 1024160138  |  **Group:** 3P14

**How to use this notebook with the starter (`UCS420_LA9_Starter.ipynb`)**

1. Run the starter notebook first (enter your roll number, it creates the tickets and prints your fingerprint).
2. Then run this notebook's cells *in the same Colab session* (or paste them below the starter's cells).
   The cell titled **"Dataset hooks"** only supplies a *stand-in* dataset when `generate_dataset` / `process_ticket`
   are **not** already defined, so it never overrides your starter. It also contains the one small adapter
   (`analyze`) that maps whatever `process_ticket()` returns onto the six counts the assignment needs — if your
   starter's keys differ, edit only that function.

Everything else uses only `tickets`, `TEST_LINES` and `process_ticket()`.

In [ ]:
print("Name: Ali Ekram | Roll No: 1024160138 | Group: 3P14")
import re, string, hashlib, random, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag, FreqDist, bigrams

pd.set_option("display.max_rows", None, "display.width", 200, "display.max_colwidth", 80)

## Dataset hooks (stand-in only used if the starter has not been run)

In [ ]:
ROLL_NUMBER = "1024160138"         # Ali Ekram, roll no. 1024160138 (used by the stand-in generator only if the starter is not loaded)

if "generate_dataset" not in globals():
    print("Starter not found in this session -> using a STAND-IN dataset (same kind of data, not your own).")
    _POOL = [
        "Hi team, my Wi-Fi isn't working since the update to version 2.0.1. Please call me at +91 98765 43210 or email john.doe@example.com. Thanks!",
        "The VPN client keeps disconnecting every 5 minutes. Logs are at https://logs.corp-it.com/vpn/session?id=4821. This is a state-of-the-art system, so it shouldn't happen!",
        "Printer on Floor 3 shows error 0x80070005. Contact helpdesk@college.edu.in or call (555) 123-4567 for a replacement toner.",
        "Battery at 95.5% after charging overnight, but the laptop shuts down randomly. Is the charger faulty? Reply to priya_s@mail.org.",
        "Cannot access the shared drive since Monday. Users are getting permission errors; see www.intranet.example.com/help for details.",
        "Outlook crashed three times today while attaching large files. Our IT admin said the patch 3.14.2 was installed yesterday.",
        "Please reset my password. I tried the self-service portal at http://accounts.example.edu/reset but it keeps loading forever.",
        "The new monitor flickers whenever I connect the HDMI cable. It isn't a driver issue because version 1.9 worked perfectly. Call 98765-43210.",
        "Email stopped syncing on my phone after the upgrade. Support asked me to write to it-support@example.com with screenshots.",
        "Wi-Fi speed dropped to 2.5 Mbps in the library. Several students are complaining; the router firmware is 4.1.0 and hasn't been updated.",
        "Application licences expired for the design lab. Unable to open Photoshop, and the licence server https://lic.example.com/status is unreachable!",
        "My keyboard types the wrong characters after installing the language pack. Ticket opened by Rahul Verma, extension 555-867-5309.",
        "The backup job failed with error code 404 on Saturday night. Storage usage is 87.3 percent; please clean old snapshots.",
        "Zoom meetings freeze every few minutes. I am using a state-of-the-art webcam, so the camera is not the problem. Contact: ravi.k@example.org.",
        "Unable to install Python 3.11.4 on the lab machines; the installer says administrator rights are required. Please escalate.",
        "A suspicious email arrived from billing@unknown-site.net asking for credentials. I didn't click anything. Is this phishing?",
        "Database queries are running slowly since the index rebuild. Average latency increased from 120.5 ms to 480.2 ms.",
        "Our website shows an expired certificate warning. Visit https://www.example.com/secure to reproduce. Urgent, please update it today.",
        "Touchpad scrolling isn't smooth after Windows 11 update. Everyone in office 12 has the same issue; call +1-800-555-0199.",
        "Software installation requests take too long. Approvals are pending for 7 days, and managers aren't responding to emails.",
    ]

    def generate_dataset(roll):
        seed = int(hashlib.sha256(str(roll).encode()).hexdigest(), 16)
        rnd = random.Random(seed)
        data = rnd.sample(_POOL, rnd.randint(12, 16))
        print("Fingerprint:", hashlib.sha256((str(roll) + "|".join(data)).encode()).hexdigest()[:8])
        return data

    _STOP = set(stopwords.words("english"))

    def _is_punct(tok):
        return all(ch in string.punctuation for ch in tok)

    def process_ticket(text):
        toks = word_tokenize(text)
        punct = [t for t in toks if _is_punct(t)]
        stops = [t for t in toks if t.lower() in _STOP]
        final = [t.lower() for t in toks
                 if not _is_punct(t) and t.lower() not in _STOP and not t.startswith("'") and t != "n't"]
        return {"sentences": sent_tokenize(text), "tokens": toks, "punctuation": punct,
                "stopwords": stops, "final_tokens": final}

    TEST_LINES = [
        "Wi-Fi isn't working; the state-of-the-art router scored 95.5 on version 2.0.1.",
        "Email john.doe@example.com or visit https://support.example.com/reset?id=42 today.",
        "Call +91 98765 43210 or (555) 123-4567; fallback: 555-987-6543.",
        "Windows 11 update KB5034441 failed with error 0x80070005 at 3.5 GB.",
        "Apple, Eagle and umbrella start with vowels; Printer and Network are Capitalized.",
        "Visit www.college.edu/portal. Then ping admin@college.edu.in!",
    ]
    tickets = generate_dataset(ROLL_NUMBER)

print(f"{len(tickets)} tickets loaded, {len(TEST_LINES)} test lines.")


def _n(x):
    """count of a list-like, or the number itself"""
    return x if isinstance(x, (int, np.integer)) else len(x)


def analyze(ticket):
    """ADAPTER: map process_ticket() output to the six pieces the assignment needs.
    Edit the keys here if your starter's process_ticket() names them differently."""
    r = process_ticket(ticket)
    return {
        "sentences": r["sentences"],
        "tokens": r["tokens"],
        "punct": r["punctuation"],
        "stopwords": r["stopwords"],
        "final": r["final_tokens"],
    }

## Q1. Preprocessing table

In [ ]:
rows = []
for i, t in enumerate(tickets, 1):
    r = analyze(t)
    rows.append({
        "ticket": i,
        "sentences": _n(r["sentences"]),
        "tokens": len(r["tokens"]),
        "punctuation": len(r["punct"]),
        "stopwords": len(r["stopwords"]),
        "unique_tokens": len({w.lower() for w in r["tokens"]}),
        "final_tokens": len(r["final"]),
    })
df1 = pd.DataFrame(rows).set_index("ticket")
df1.loc["TOTAL"] = df1.sum()
print(df1)
all_unique = len({w.lower() for t in tickets for w in analyze(t)["tokens"]})
print(f"\n(Note: the TOTAL of 'unique_tokens' is the sum of per-ticket counts; unique tokens across the whole corpus = {all_unique}.)")

In [ ]:
# one selected ticket: show the actual tokens, punctuation tokens, stopwords and final tokens
sel = 0
r = analyze(tickets[sel])
print(f"Selected ticket #{sel + 1}:\n{tickets[sel]}\n")
print("tokens       :", r["tokens"])
print("punctuation  :", r["punct"])
print("stopwords    :", r["stopwords"])
print("final tokens :", r["final"])

## Q2. Regex extraction and replacement

In [ ]:
# ---- extraction patterns -------------------------------------------------------------------------------------
LONG_WORD = r"\b[A-Za-z]{6,}(?:['’][a-z]+)?\b"           # words with more than 5 letters (a contraction such as shouldn't stays one word)
NUMBER = r"\b\d+(?:\.\d+)*\b"                           # 42, 95.5, 2.0.1 stay as ONE match each
CAPITALIZED = r"\b[A-Z][a-z]*(?:['’][a-z]+)?\b"          # starts with a capital: Printer, I'm (all-caps acronyms like VPN are not "capitalized words")
VOWEL_WORD = r"\b[AEIOUaeiou][A-Za-z]*(?:['’][a-z]+)?\b"  # words beginning with a vowel: error, isn't

# ---- replacement patterns (also reused by the tokenizer in Q3) ---------------------------------------------------
URL = r"(?:https?://|ftp://|(?<![@\w.-])www\.)[^\s<>\"']*[^\s<>\"'.,;:!?)\]}]"
EMAIL = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+"
BARE_DOMAIN = (r"(?<![@\w./-])(?:[A-Za-z0-9-]+\.)+(?:com|org|net|edu|gov|io|co|in)\b"
               r"(?:/[^\s<>\"']*[^\s<>\"'.,;:!?)\]}])?")
PHONE = (r"(?<![\w.+-])(?:"
         r"\+\d{1,3}[\s.-]?(?:\(\d{1,4}\)[\s.-]?)?\d{2,5}(?:[\s.-]?\d{2,5}){1,3}"   # +91 98765 43210, +1-800-555-0199
         r"|\(\d{2,4}\)[\s.-]?\d{3,5}[\s.-]?\d{3,5}"                                 # (555) 123-4567
         r"|\d{3}[.-]\d{3}[.-]\d{4}"                                                 # 555-123-4567
         r"|\d{5}[\s.-]\d{5}"                                                        # 98765-43210
         r"|\d{10}"                                                                  # 9876543210
         r")(?![\w-]|\.\d)")


def extract_all(text):
    return {
        "long words (>5 letters)": re.findall(LONG_WORD, text),
        "numbers": re.findall(NUMBER, text),
        "capitalized": re.findall(CAPITALIZED, text),
        "vowel-initial": re.findall(VOWEL_WORD, text),
    }


def mask(text):
    text = re.sub(URL, "<URL>", text)               # URLs first (they may contain '@' or look like e-mail/domains)
    text = re.sub(EMAIL, "<EMAIL>", text)
    text = re.sub(BARE_DOMAIN, "<URL>", text)       # bare domains such as portal.example.com/help
    text = re.sub(PHONE, "<PHONE>", text)
    return text


print("=== TEST_LINES: extraction ===")
for line in TEST_LINES:
    print("\n", line)
    for k, v in extract_all(line).items():
        print(f"   {k:24s}: {v}")

print("\n=== TEST_LINES: replacement ===")
for line in TEST_LINES:
    print("\n  before:", line)
    print("  after :", mask(line))

In [ ]:
print("=== TICKETS: extraction (first 3 tickets shown in full, totals for all) ===")
for i, t in enumerate(tickets[:3], 1):
    print(f"\nTicket {i}: {t}")
    for k, v in extract_all(t).items():
        print(f"   {k:24s}: {v}")

totals = collections.Counter()
for t in tickets:
    for k, v in extract_all(t).items():
        totals[k] += len(v)
print("\nMatches over all tickets:", dict(totals))

print("\n=== TICKETS: replacement ===")
for i, t in enumerate(tickets, 1):
    m = mask(t)
    if m != t:
        print(f"\nTicket {i} before: {t}\n         after : {m}")

**Design notes**
* `NUMBER` = `\d+(?:\.\d+)*` – the optional repeated `.digits` group keeps `95.5` and `2.0.1` as single matches,
  while a sentence-ending full stop is not swallowed (a `.` must be followed by a digit).
* `\b` on both sides of `LONG_WORD` avoids matching pieces of alphanumerics such as `Windows11`; the `{6,}` quantifier
  means "more than 5 letters".
* Replacement order matters: URLs are masked before e-mails (a URL can contain an `@`), and phone numbers last with
  look-arounds so ticket ids, versions (`2.0.1`) and decimals (`95.5`) are never mistaken for phone numbers.
* Extraction runs on the *raw* text as the assignment asks, so e-mail/URL fragments (`example`, `support`…) also count
  as ordinary words; run `mask()` first if you want them excluded.

## Q3. Custom tokenization with one `RegexpTokenizer` pattern

One alternation, ordered from most specific to least specific so that e-mails, URLs and phone numbers are claimed
*before* the generic word rule can split them:

`URL | EMAIL | bare-domain | PHONE | version/decimal | word with inner - or ' | single punctuation mark`

In [ ]:
VERSION_DECIMAL = r"[vV]?\d+(?:\.\d+)+"                 # 95.5, 2.0.1, v3.14.2
WORD = r"\w+(?:[-'’]\w+)*"                             # isn't, Wi-Fi, state-of-the-art, 0x80070005, plain integers
PUNCT = r"\.{3}|[^\w\s]"                               # '...' or any single remaining symbol

TOKEN_PATTERN = "|".join([URL, EMAIL, BARE_DOMAIN, PHONE, VERSION_DECIMAL, WORD, PUNCT])
_rt = RegexpTokenizer(TOKEN_PATTERN)


def my_tokenize(text):
    return _rt.tokenize(text)


expected = ["isn't", "Wi-Fi", "state-of-the-art", "95.5", "2.0.1"]
print("sanity:", my_tokenize("Wi-Fi isn't state-of-the-art: 95.5 / 2.0.1"))

In [ ]:
print("=== TEST_LINES ===")
for line in TEST_LINES:
    print("\n", line)
    print("  my_tokenize :", my_tokenize(line))
    print("  word_tokenize:", word_tokenize(line))

In [ ]:
print("=== TICKETS (first 4 shown) ===")
for i, t in enumerate(tickets[:4], 1):
    print(f"\nTicket {i}: {t}")
    print("  my_tokenize :", my_tokenize(t))
    print("  word_tokenize:", word_tokenize(t))

In [ ]:
# automatic comparison: tokens produced by only one of the tokenizers
diffs = []
for src, line in [("TEST", l) for l in TEST_LINES] + [("TICKET", t) for t in tickets]:
    mine, nl = my_tokenize(line), word_tokenize(line)
    only_mine = sorted(set(mine) - set(nl))
    only_nltk = sorted(set(nl) - set(mine))
    if only_mine or only_nltk:
        diffs.append({"source": src, "text": line[:55] + "…", "only my_tokenize": only_mine, "only word_tokenize": only_nltk})
diff_df = pd.DataFrame(diffs)
print(diff_df.to_string())

### Differences between `my_tokenize` and `word_tokenize` (at least three)

`word_tokenize` is a general-purpose Penn-Treebank-style tokenizer; mine is tuned for IT-support text.

| # | Case | `my_tokenize` | `word_tokenize` | Why they differ |
|---|------|---------------|-----------------|-----------------|
| 1 | **Contraction** `isn't` | `isn't` (one token) | `is`, `n't` | The Treebank convention splits clitics so that `n't` can be analysed as the negation `not`; my `WORD` rule allows an inner apostrophe, keeping the surface word intact. |
| 2 | **E-mail** `john.doe@example.com` | one token | `john.doe`, `@`, `example.com` | `word_tokenize` always splits off `@`; my `EMAIL` alternative matches the whole address first. |
| 3 | **URL** `https://support.example.com/reset?id=42` | one token | several tokens (the scheme, `:`, the rest of the address, `?`, …) | The Treebank rules split on `:` and `?`; a URL is meaningless once broken, so my `URL` alternative keeps it whole. |
| 4 | **Phone** `+91 98765 43210`, `(555) 123-4567` | one token | several tokens (e.g. `+`, `91`, `98765`, `43210`) | Whitespace, `+` and brackets separate tokens for `word_tokenize`; my `PHONE` alternative treats the number as one entity. |
| 5 | Sentence-final full stop after `2.0.1.` | `2.0.1`, `.` | `2.0.1`, `.` (same) | Both keep the version intact — the version rule only allows `.` when followed by a digit. |

Same behaviour for **hyphenated words** (`Wi-Fi`, `state-of-the-art`) and **decimals** (`95.5`): both tokenizers keep them
whole. The table above (`diff_df`) lists the concrete differences found in *your* tickets.
For classification or NER on support tickets my output is better (entities stay intact); for linguistic analysis
(POS tagging, parsing) the Treebank output is the standard that downstream tools expect.

## Q4. Stemming and lemmatization on the unique final tokens

In [ ]:
final_all = [w for t in tickets for w in analyze(t)["final"]]
unique_final = sorted(set(final_all))
print(len(final_all), "final tokens ->", len(unique_final), "unique")

# ---- Regexp stemmer: choose the suffix from the data ------------------------------------------------------
candidates = ["ing", "ed", "ment", "ion", "ly", "ness", "er"]
support = {s: [w for w in unique_final if w.endswith(s) and len(w) >= len(s) + 3] for s in candidates}
for s, ws in support.items():
    print(f"suffix -{s:5s}: {len(ws)} words {ws[:6]}")
best_suffix = max(support, key=lambda s: len(support[s]))
print(f"\nChosen suffix: -{best_suffix}  (evidence: {support[best_suffix]})")

regexp_stemmer = RegexpStemmer(best_suffix + "$", min=len(best_suffix) + 3)

**Why this suffix?** The suffix with the most evidence among the unique final tokens of *this* dataset is chosen
automatically (printed above together with the words that justify it — at least two of them, e.g. words like the ones
listed). A pattern anchored with `$` only strips the suffix at the end of a word, and `min=len(suffix)+3` stops very
short words from being destroyed (otherwise `ring` → `r`). The limitation of such a stemmer is that it is blind:
it strips the suffix even when it is part of the root (e.g. `string` → `str`) and cannot handle irregular forms.

In [ ]:
porter, lancaster, wnl = PorterStemmer(), LancasterStemmer(), WordNetLemmatizer()


def to_wn(tag):
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)


# POS tags are obtained IN CONTEXT: tag each raw ticket, then give every final token its most frequent tag
tag_counts = collections.defaultdict(collections.Counter)
for t in tickets:
    toks = word_tokenize(t)
    for w, tag in pos_tag(toks):
        tag_counts[w.lower()][tag] += 1


def tag_of(w):
    if w in tag_counts:
        return tag_counts[w].most_common(1)[0][0]
    return pos_tag([w])[0][1]


rows = []
for w in unique_final:
    tag = tag_of(w)
    rows.append({"word": w, "pos": tag, "porter": porter.stem(w), "lancaster": lancaster.stem(w),
                 "regexp": regexp_stemmer.stem(w), "lemma": wnl.lemmatize(w, to_wn(tag))})
all_df = pd.DataFrame(rows)

methods = ["porter", "lancaster", "regexp", "lemma"]
differ = all_df[all_df[methods].nunique(axis=1) > 1].reset_index(drop=True)
print(f"{len(differ)} of {len(all_df)} words get different outputs from the four methods:\n")
print(differ.to_string())

## Q5. Corpus statistics

In [ ]:
def is_punct(tok):
    return all(ch in string.punctuation for ch in tok)


S1 = [w.lower() for t in tickets for w in analyze(t)["tokens"]]          # all tokens
S2 = [w for w in S1 if not is_punct(w)]                                    # without punctuation
S3 = final_all                                                             # final tokens
S4 = [porter.stem(w) for w in S3]                                          # S3 after Porter stemming

stats = []
for name, S in [("S1: all tokens", S1), ("S2: no punctuation", S2), ("S3: final tokens", S3), ("S4: S3 + Porter", S4)]:
    stats.append({"set": name, "total tokens": len(S), "unique tokens": len(set(S)), "TTR": round(len(set(S)) / len(S), 4)})
stats_df = pd.DataFrame(stats).set_index("set")
print(stats_df)

In [ ]:
print("Top 10 words of S1:", FreqDist(S1).most_common(10))
print("Top 10 words of S3:", FreqDist(S3).most_common(10))

In [ ]:
# Zipf plot for S3: log(rank) vs log(frequency)
freqs = np.array(sorted(FreqDist(S3).values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
slope, intercept = np.polyfit(np.log(ranks), np.log(freqs), 1)

plt.figure(figsize=(6, 4))
plt.plot(np.log(ranks), np.log(freqs), "o-", label="S3 data")
plt.plot(np.log(ranks), intercept + slope * np.log(ranks), "--", label=f"fit: slope = {slope:.2f}")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Zipf plot of S3 (final tokens)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Fitted slope = {slope:.2f} (Zipf's law predicts a line close to -1; small corpora deviate, especially in the long tail of words that occur once).")

In [ ]:
# top 5 bigrams: counted inside each ticket so that no bigram spans two different tickets
def top_bigrams(token_lists, k=5):
    c = collections.Counter()
    for toks in token_lists:
        c.update(bigrams(toks))
    return c.most_common(k)


s1_by_ticket = [[w.lower() for w in analyze(t)["tokens"]] for t in tickets]
s3_by_ticket = [analyze(t)["final"] for t in tickets]
print("Top 5 bigrams of S1:", top_bigrams(s1_by_ticket))
print("Top 5 bigrams of S3:", top_bigrams(s3_by_ticket))
print("(S3 bigrams are adjacent *remaining* words, i.e. after punctuation and stopwords were removed.)")

## Q6. Interpretation
The text below is generated from the numbers of *your* dataset, so it stays consistent with your tickets.

In [ ]:
def trend(a, b):
    return "rises" if b > a else "falls" if b < a else "stays the same"


t = stats_df["TTR"]
print("(a) Why does TTR change from S1 to S4?")
print(f"   TTR: S1 {t.iloc[0]} -> S2 {t.iloc[1]} -> S3 {t.iloc[2]} -> S4 {t.iloc[3]}")
print(f"   * S1 -> S2: TTR {trend(t.iloc[0], t.iloc[1])}. Punctuation marks ({stats_df.iloc[0,0] - stats_df.iloc[1,0]} tokens) are few distinct "
      f"types repeated very often, so removing them shrinks the token count a lot but the vocabulary only a little.")
print(f"   * S2 -> S3: TTR {trend(t.iloc[1], t.iloc[2])}. Stopwords such as 'the', 'is', 'to' are extremely repetitive; "
      f"dropping them removes many repeated tokens while keeping the content words, which are mostly distinct.")
print(f"   * S3 -> S4: TTR {trend(t.iloc[2], t.iloc[3])}. Stemming keeps the number of tokens identical but merges inflected forms "
      f"(unique types {stats_df.iloc[2,1]} -> {stats_df.iloc[3,1]}), so the numerator shrinks and TTR can only stay equal or fall.")

In [ ]:
# (b) one example where word_tokenize and my_tokenize differ
example = None
for line in TEST_LINES + tickets:
    mine, nl = my_tokenize(line), word_tokenize(line)
    odd = [tok for tok in mine if tok not in nl]
    if odd:
        example = (line, odd[0], mine, nl)
        break
line, tok, mine, nl = example
if "@" in tok:
    why = "my pattern recognises an e-mail address as a single entity, while word_tokenize splits off the '@' symbol."
elif tok.startswith(("http", "www")) or "/" in tok:
    why = "my pattern keeps a URL whole, while word_tokenize splits it at ':' / '?' and other punctuation."
elif "'" in tok:
    why = "my pattern keeps a contraction as one word, while word_tokenize splits it into 'is' + \"n't\" (Treebank convention)."
elif any(ch.isdigit() for ch in tok) and any(ch in tok for ch in "+()- "):
    why = "my pattern keeps a phone number as one token, while word_tokenize splits it on spaces, '+' and brackets."
else:
    why = "my pattern keeps this unit together, while word_tokenize applies its general Treebank splitting rules."
print("(b) Example:", line)
print("    my_tokenize   ->", mine)
print("    word_tokenize ->", nl)
print(f"    Difference: token {tok!r} - {why}")

In [ ]:
# (c) one example where stemming gives an undesirable result
by_stem = collections.defaultdict(set)
for w in unique_final:
    by_stem[porter.stem(w)].add(w)
conflations = [(s, ws) for s, ws in by_stem.items() if len({wnl.lemmatize(w) for w in ws}) > 1]
non_words = [(w, porter.stem(w)) for w in unique_final
             if len(w) > 4 and porter.stem(w) != w and not wordnet.synsets(porter.stem(w))]
print("(c) Undesirable stemming examples in this dataset:")
if conflations:
    s_, ws = conflations[0]
    print(f"    * OVER-STEMMING: the distinct words {sorted(ws)} all collapse to the single stem '{s_}', so the distinction between them is lost"
          + ("" if wordnet.synsets(s_) else f" (and '{s_}' is not even a dictionary word)") + ".")
if non_words:
    w, s = non_words[0]
    print(f"    * NON-WORD STEM: Porter turns '{w}' into '{s}', which is not a dictionary word, so the result is hard to read/interpret.")
lan_bad = [(w, lancaster.stem(w)) for w in unique_final if len(lancaster.stem(w)) <= 2 and len(w) > 4]
if lan_bad:
    print(f"    * AGGRESSIVE: Lancaster reduces '{lan_bad[0][0]}' to '{lan_bad[0][1]}' - almost all information is lost.")

In [ ]:
# (d) POS-aware lemmatization preserves meaning better than stemming
cands = [r for r in rows if r["lemma"] != r["word"] and r["porter"] != r["lemma"] and wordnet.synsets(r["lemma"])]
print("(d) Examples in this dataset where the POS-aware lemma is a proper word and the stem is not:")
for r in cands[:5]:
    print(f"    {r['word']!r} (tag {r['pos']}): lemma -> {r['lemma']!r}   vs   Porter -> {r['porter']!r}")
print("\n    Classic illustration (needs the POS tag):")
print("    wnl.lemmatize('better', 'a') ->", wnl.lemmatize("better", "a"), "| Porter('better') ->", porter.stem("better"))
print("    wnl.lemmatize('was', 'v')    ->", wnl.lemmatize("was", "v"), "| Porter('was')    ->", porter.stem("was"))
print("    wnl.lemmatize('meeting','v') ->", wnl.lemmatize("meeting", "v"), "| wnl.lemmatize('meeting','n') ->", wnl.lemmatize("meeting", "n"))

**Summary of the answers**
* **(a)** TTR rises when punctuation (S1→S2) and stopwords (S2→S3) are removed because those tokens are highly repetitive;
  it falls from S3→S4 because stemming merges different surface forms into one type without changing the token count.
* **(b)** The code above prints a concrete example from your data; the usual ones are contractions (`isn't`), e-mail
  addresses, URLs and phone numbers, which stay whole in `my_tokenize` but are split by `word_tokenize`.
* **(c)** Stemmers cut strings, not meanings: they can produce non-words (`issues → issu`) or merge unrelated words
  (over-stemming, e.g. `universe/university → univers`), and Lancaster is even more aggressive.
* **(d)** A POS-aware lemmatizer returns real dictionary forms: `better` (adjective) → `good`, `was` (verb) → `be`,
  `issues` (noun) → `issue`, while stemming gives `better`, `wa`, `issu`.